###Load Env Variables
###kernal and interpreter is python

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

#### importing llm packages
#### DOTENV

In [2]:
# from langchain_openai import OpenAI
# llm=OpenAI(model='gpt-4o')
# res=llm.invoke('who is pm of usa?')
# print(res.content)

In [2]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    api_key="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")


res = llm.invoke("who is pm of japan?")
print(res.content)



ModuleNotFoundError: No module named 'langchain_nvidia_ai_endpoints'

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
res=llm.invoke("whats your name?")#.....this is not a actual method for input thus we use prompting




In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='qwen/qwen3.6-27b')
prompt=[
    {'role':'system','content':'you are a web developer'},
    {'role':'user','content':'how to animate the website'}
]
print(llm.invoke(prompt).content)



<think>
Here's a thinking process:

1.  **Understand User Query:** The user asks "how to animate the website". This is a broad question about web animation techniques. I need to provide a comprehensive, structured, and practical guide covering different approaches, tools, and best practices.

2.  **Identify Key Areas of Web Animation:**
   - CSS Animations/Transitions
   - JavaScript Animations (Vanilla JS, Libraries like GSAP, Framer Motion)
   - SVG Animations
   - Canvas/WebGL (for advanced/complex animations)
   - Framework-specific solutions (React, Vue, Angular)
   - Best practices & performance considerations

3.  **Structure the Response:**
   - Introduction: Briefly explain why web animation matters and the main approaches
   - Method 1: CSS Animations & Transitions (Beginner-friendly, performant)
   - Method 2: JavaScript Libraries (Advanced, flexible)
   - Method 3: Framework-Specific Solutions (Modern web dev)
   - Method 4: Advanced/Complex Animations (SVG, Canvas, WebGL)

###prompting


In [ ]:
from langchain_openai import OpenAI
llm=OpenAI(model="gpt-4o")
prompts=[
    {'role':'system','content':'you are python developer'},#here we are telling the role to our system
    {'role':'user','content':'how to sort array? '}
]#create the "<list of dictionary>" with keys user(role)/system(role) and values(content) as input  
res=llm.invoke(prompts)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(model="gemini-3")
prompt=[
    {'role':'system','content':'you are a language translater'},
    {'role':'user','content':'i love cars'}
]
res=llm.invoke(prompt)


#### PROMPT TEMPLATES-DYNAMIC PROMPTS
#requirnments:
1.from langchain_core.prompts import ChatPromptTemplate
2.from langchain_core.output_parsers import StrOutputParser

In [18]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm=ChatGroq(model='qwen/qwen3.6-27b')
out=StrOutputParser()#............it extracts raw output from llm model

prompts=ChatPromptTemplate.from_messages(
    [{'role':'system','content':'you are a developer{coding_language}'},
     {'role':'user','content':'{query}'}])

finalprompt=prompts.format_messages(coding_language='cpp',query='sorting of array')
res=llm.invoke(finalprompt)

chain= prompts | llm | out
res=chain.invoke({'coding_language':'cpp','query':'sorting of an array'})

print(res)




<think>
Here's a thinking process:

1.  **Understand User Input:** The user said "sorting of an array". This is a very broad topic in C++ programming. They likely want to know how to sort an array in C++, possibly with code examples, explanations of different methods, and best practices.

2.  **Identify Key Concepts:**
   - C++ provides built-in sorting functions in the `<algorithm>` header (`std::sort`)
   - Manual sorting algorithms (Bubble, Selection, Insertion, Merge, Quick, Heap)
   - Differences between arrays and vectors
   - Time complexity considerations
   - Custom comparators/lambdas
   - Best practices

3.  **Determine Response Structure:**
   - Start with the simplest/most recommended approach (`std::sort`)
   - Provide clear code examples
   - Explain manual algorithms briefly (when/why to use them)
   - Cover edge cases/custom sorting
   - Mention performance/complexity
   - Keep it concise and C++ focused

4.  **Draft - Section by Section:**
   *(Mental Refinement)*
  

###chaining.......

In [ ]:
chain= prompts | llm | out
res=chain.invoke({'coding_language':'cpp','query':'sorting of an array'})


###adding memory to llm

In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='qwen/qwen3.6-27b')


history=[]
while True:
   query=input('user:')
   if query.lower() in ['end','stop','delete']:
       
      print('history deleted')
      break
 
   history.append({'role':'user','content':query})
   print('user:',query)
   res=llm.invoke(history)
   history.append({'role':'ai','content':res.content})
   print('ai:',res.content,'\n')


###structured output
from pydentic and typing libraries

In [36]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='qwen/qwen3.6-27b')
text="my name is om divekar with roll no:103 and age:20"

#defining structure 
from pydantic import BaseModel, Field

class ResponseStructure(BaseModel):
    name:str=Field(description='complete name')
    roll:int=Field(description='roll no')
    age:int=Field(description='age')
#now we are able to get the output in above format/structure....
structured_llm=llm.with_structured_output(ResponseStructure)
# res=structured_llm.invoke(f"plz give me name, age and roll no")

res=structured_llm.invoke(text)

In [ ]:
res.model_dump()
# res.roll
# res.name



103

In [46]:
from pydantic import BaseModel, Field
from typing import List

class MovieRecommendation(BaseModel):
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    genre: str = Field(description="Genre of the movie")

class MovieList(BaseModel):
    movies: List[MovieRecommendation] = Field(description="List of recommended movies")

movies_llm=llm.with_structured_output(MovieList)

In [51]:
res=movies_llm.invoke("Suggest me interesting movies")

In [52]:
res.model_dump()

{'movies': [{'title': 'Inception', 'year': 2010, 'genre': 'Sci-Fi'},
  {'title': 'The Shawshank Redemption', 'year': 1994, 'genre': 'Drama'},
  {'title': 'Spirited Away', 'year': 2001, 'genre': 'Animation'},
  {'title': 'Parasite', 'year': 2019, 'genre': 'Thriller'},
  {'title': 'The Grand Budapest Hotel', 'year': 2014, 'genre': 'Comedy'}]}

#### Implementation of OLLAMA

In [3]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model='gemma3:1b')
query=input('ask?')
res=llm.invoke([{'role':'user','content':query}])
print(res.content)

ModuleNotFoundError: No module named 'langchain_ollama'